# Transacciones y ACID

BEGIN, COMMIT, ROLLBACK, SAVEPOINT, niveles de aislamiento y prevención de deadlocks

## Introducción

> Una transacción es un conjunto de operaciones SQL que se ejecutan como una unidad indivisible: o todas tienen éxito, o ninguna se aplica. Las propiedades ACID (Atomicity, Consistency, Isolation, Durability) garantizan la integridad de los datos incluso ante fallos del sistema o accesos concurrentes. Son el pilar de cualquier sistema que maneje datos críticos: transferencias bancarias, reservas de vuelos, pedidos en e-commerce. Entender cómo funcionan las transacciones y cómo controlarlas con BEGIN / COMMIT / ROLLBACK / SAVEPOINT es esencial para escribir aplicaciones robustas.

### Objetivos de Aprendizaje

- Comprender las cuatro propiedades ACID y por qué son necesarias
- Usar BEGIN, COMMIT y ROLLBACK para controlar transacciones explícitas
- Aplicar SAVEPOINT para rollback parcial dentro de una transacción
- Conocer los niveles de aislamiento y qué anomalías previenen
- Identificar y prevenir deadlocks en operaciones concurrentes

## Propiedades ACID

> ACID es el acrónimo que define qué debe garantizar cualquier sistema de base de datos transaccional. Atomicity: la transacción es todo o nada — si falla a mitad, todos los cambios se revierten. Consistency: la BD siempre pasa de un estado válido a otro válido (restricciones, claves foráneas, etc.). Isolation: las transacciones concurrentes no se ven entre sí hasta que se confirman. Durability: una vez hecho COMMIT, los datos persisten aunque el sistema falle. Analogía real: una transferencia bancaria debe debitar UNA cuenta y acreditar OTRA de forma atómica — no puede haber un estado intermedio donde el dinero desaparezca.


In [ ]:
import sqlite3

# Analogía: transferencia bancaria
# Sin transacción → riesgo de inconsistencia
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE cuentas (
    id INTEGER PRIMARY KEY,
    titular TEXT NOT NULL,
    saldo REAL NOT NULL CHECK(saldo >= 0)
)''')
cursor.executemany('INSERT INTO cuentas VALUES (?,?,?)', [
    (1, 'Alice', 1000.00),
    (2, 'Bob',    500.00),
])
conn.commit()

# ATOMICITY: ambas operaciones deben ocurrir juntas
# Si solo el débito ocurre → inconsistencia
# CONSISTENCY: CHECK(saldo >= 0) se respeta siempre
# ISOLATION: otra conexión no ve el estado intermedio
# DURABILITY: tras COMMIT, datos sobreviven a un crash

print("Estado inicial:")
for row in cursor.execute("SELECT titular, saldo FROM cuentas"):
    print(f"  ${row[0]:<8}: $${row[1]:,.2f}")

# Verificar que la restricción funciona
try:
    cursor.execute("UPDATE cuentas SET saldo = -100 WHERE id = 1")
except sqlite3.IntegrityError as e:
    print(f"\nConstraint OK → {e}")
    conn.rollback()

print("\nLas propiedades ACID protegen la integridad de los datos.")

## BEGIN / COMMIT / ROLLBACK

> En Python con sqlite3, el módulo maneja transacciones automáticamente según isolation_level. Con isolation_level=None (autocommit) cada sentencia es su propia transacción. Para control explícito: conn.execute("BEGIN") o conn.isolation_level != None inicia una transacción; conn.commit() la confirma; conn.rollback() la deshace. El patrón recomendado es usar try/except/finally para garantizar que siempre se haga commit o rollback.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE cuentas (
    id INTEGER PRIMARY KEY,
    titular TEXT NOT NULL,
    saldo REAL NOT NULL CHECK(saldo >= 0)
)''')
conn.commit()
cursor.executemany('INSERT INTO cuentas VALUES (?,?,?)', [
    (1, 'Alice', 1000.00),
    (2, 'Bob',    500.00),
])
conn.commit()

def transferir(conn, id_origen, id_destino, monto):
    """Transferencia atómica con BEGIN/COMMIT/ROLLBACK."""
    cursor = conn.cursor()
    try:
        # Iniciar transacción explícita
        conn.execute("BEGIN")

        # 1. Verificar saldo suficiente
        saldo = cursor.execute(
            "SELECT saldo FROM cuentas WHERE id = ?", (id_origen,)
        ).fetchone()[0]
        if saldo < monto:
            raise ValueError(f"Saldo insuficiente: ${saldo:.2f} < ${monto:.2f}")

        # 2. Débito
        cursor.execute(
            "UPDATE cuentas SET saldo = saldo - ? WHERE id = ?",
            (monto, id_origen)
        )

        # 3. Crédito
        cursor.execute(
            "UPDATE cuentas SET saldo = saldo + ? WHERE id = ?",
            (monto, id_destino)
        )

        conn.commit()  # confirmar ambas operaciones
        print(f"✅ Transferencia de ${monto:.2f} exitosa")

    except Exception as e:
        conn.rollback()  # deshacer todo si algo falla
        print(f"❌ Error: {e} — transacción revertida")

# Caso 1: transferencia válida
transferir(conn, 1, 2, 300.00)
for row in cursor.execute("SELECT titular, saldo FROM cuentas"):
    print(f"   ${row[0]}: $${row[1]:.2f}")

# Caso 2: saldo insuficiente → rollback
print()
transferir(conn, 2, 1, 1000.00)
for row in cursor.execute("SELECT titular, saldo FROM cuentas"):
    print(f"   ${row[0]}: $${row[1]:.2f}")

## SAVEPOINT — Rollback Parcial
> SAVEPOINT permite crear puntos de guardado dentro de una transacción. Con ROLLBACK TO nombre_savepoint se deshacen solo las operaciones desde ese punto, sin cancelar toda la transacción. RELEASE nombre_savepoint libera el savepoint (pero no confirma la transacción). Es el mecanismo para implementar "transacciones anidadas" en SQL y es especialmente útil en operaciones batch donde algunos grupos pueden fallar sin afectar al resto.


In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL UNIQUE,
    precio REAL NOT NULL CHECK(precio > 0),
    categoria TEXT NOT NULL
)''')
conn.commit()

# Batch insert con SAVEPOINT: rollback solo del grupo fallido
grupos = [
    [("Laptop Pro", 1299.99, "Electronics"),
     ("Monitor 27", 349.00, "Electronics"),
     ("Teclado RGB", 89.99, "Electronics")],   # OK

    [("Silla Ergonómica", 299.00, "Furniture"),
     ("Lámpara LED", -15.00, "Furniture"),      # ← precio negativo = falla
     ("Escritorio", 449.00, "Furniture")],

    [("Auriculares BT", 129.99, "Electronics"),
     ("Webcam HD", 79.99, "Electronics"),
     ("Hub USB-C", 49.99, "Electronics")],      # OK

    [("Mouse Inalámbrico", 39.99, "Electronics"),
     ("Mouse Inalámbrico", 39.99, "Electronics"), # ← duplicado = falla
     ("Alfombrilla XL", 24.99, "Electronics")],
]

conn.execute("BEGIN")
exitosos = 0
fallidos = 0

for i, grupo in enumerate(grupos):
    sp = f"sp_grupo_{i}"
    conn.execute(f"SAVEPOINT {sp}")
    try:
        cursor.executemany(
            "INSERT INTO productos (nombre, precio, categoria) VALUES (?,?,?)",
            grupo
        )
        conn.execute(f"RELEASE {sp}")  # confirmar este grupo
        exitosos += len(grupo)
        print(f"✅ Grupo {i+1}: {len(grupo)} productos insertados")
    except Exception as e:
        conn.execute(f"ROLLBACK TO {sp}")
        conn.execute(f"RELEASE {sp}")
        fallidos += len(grupo)
        print(f"❌ Grupo {i+1} revertido: {e}")

conn.commit()

total = cursor.execute("SELECT COUNT(*) FROM productos").fetchone()[0]
print(f"\nResultado: {total} productos en BD ({exitosos} OK, {fallidos} revertidos)")

## Niveles de Aislamiento

> Los niveles de aislamiento controlan qué tan aisladas están las transacciones concurrentes entre sí. De menor a mayor aislamiento: READ UNCOMMITTED (ve cambios no confirmados = dirty reads), READ COMMITTED (solo ve confirmados, pero puede haber non-repeatable reads), REPEATABLE READ (el mismo SELECT siempre devuelve los mismos rows, pero puede haber phantom reads), SERIALIZABLE (aislamiento total, como si las tx fueran secuenciales). SQLite no usa estos niveles estándar; en su lugar usa DEFERRED (por defecto), IMMEDIATE (lock al escribir), y EXCLUSIVE (lock exclusivo total). PostgreSQL y MySQL sí soportan los cuatro niveles estándar.

In [ ]:
import sqlite3

# SQLite: BEGIN [DEFERRED | IMMEDIATE | EXCLUSIVE]
# PostgreSQL/MySQL: SET TRANSACTION ISOLATION LEVEL ...

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE inventario (
    producto TEXT PRIMARY KEY,
    stock INTEGER NOT NULL CHECK(stock >= 0)
)''')
cursor.executemany('INSERT INTO inventario VALUES (?,?)', [
    ('Laptop', 10), ('Mouse', 50), ('Teclado', 30)
])
conn.commit()

# DEFERRED (default): adquiere locks cuando hace primer acceso
conn2 = sqlite3.connect(':memory:')
# Attach same DB — demo conceptual con dos conexiones no se puede en :memory:
# En producción con archivo real:

print("Tipos de anomalías que los niveles previenen:")
print()
anomalias = [
    ("Dirty Read",         "Lee cambios de otra tx NO confirmada aún",
     "READ UNCOMMITTED"),
    ("Non-Repeatable Read","Mismo SELECT da resultados distintos en la tx",
     "READ COMMITTED"),
    ("Phantom Read",       "Nuevas filas aparecen entre dos SELECTs de la tx",
     "REPEATABLE READ"),
    ("Lost Update",        "Dos tx leen, modifican y escriben el mismo dato",
     "SERIALIZABLE"),
]
for nombre, desc, min_nivel in anomalias:
    print(f"  ⚠️  {nombre}")
    print(f"      {desc}")
    print(f"      Prevenido desde: {min_nivel}")
    print()

# SQLite — modos de transacción
modos = [
    ("DEFERRED",  "Adquiere lock de lectura al primer READ, escritura al primer WRITE"),
    ("IMMEDIATE", "Adquiere lock de escritura al BEGIN (previene escrituras concurrentes)"),
    ("EXCLUSIVE", "Lock total: nadie puede leer ni escribir mientras dura"),
]
print("Modos de transacción en SQLite:")
for modo, desc in modos:
    print(f"  BEGIN {modo:<10} → {desc}")

# -- PostgreSQL/MySQL (comentario referencia) --
# conn.execute("SET TRANSACTION ISOLATION LEVEL SERIALIZABLE")
# conn.execute("SET TRANSACTION ISOLATION LEVEL REPEATABLE READ")
# conn.execute("SET TRANSACTION ISOLATION LEVEL READ COMMITTED")
# conn.execute("SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED")

## Deadlocks y Cómo Prevenirlos

Un deadlock (bloqueo mutuo) ocurre cuando dos o más transacciones se bloquean mutuamente esperando locks que la otra tiene. Tx A tiene lock en tabla1 y espera lock en tabla2; Tx B tiene lock en tabla2 y espera lock en tabla1 — ambas esperan para siempre. Detección vs timeout: los SGBD como PostgreSQL detectan deadlocks automáticamente y matan a una de las transacciones; MySQL usa timeout configurable (innodb_lock_wait_timeout). Prevención: 1) Siempre acceder a tablas en el mismo orden en todas las transacciones, 2) Mantener las transacciones cortas, 3) Usar SELECT ... FOR UPDATE para declarar intención de escritura, 4) Evitar transacciones de larga duración.


In [ ]:
import sqlite3
import threading
import time

# Demostración de deadlock potencial y su prevención
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.executescript('''
    CREATE TABLE cuenta_a (id INTEGER PRIMARY KEY, saldo REAL);
    CREATE TABLE cuenta_b (id INTEGER PRIMARY KEY, saldo REAL);
    INSERT INTO cuenta_a VALUES (1, 1000.0);
    INSERT INTO cuenta_b VALUES (1, 1000.0);
''')
conn.commit()

print("=== PATRÓN QUE CAUSA DEADLOCK ===")
print()
print("  Tx1: LOCK cuenta_a → espera cuenta_b")
print("  Tx2: LOCK cuenta_b → espera cuenta_a")
print("  → Deadlock: ambas esperan indefinidamente")
print()

print("=== PREVENCIÓN: orden consistente de acceso ===")
print()
print("  Regla: SIEMPRE acceder a tablas en orden alfabético")
print("  Tx1: LOCK cuenta_a → LOCK cuenta_b (orden: a, b)")
print("  Tx2: LOCK cuenta_a → LOCK cuenta_b (mismo orden: a, b)")
print("  → No deadlock: Tx2 espera a que Tx1 libere cuenta_a")
print()

# Estrategias de prevención
estrategias = [
    ("Orden consistente",     "Acceder siempre a recursos en el mismo orden global"),
    ("Transacciones cortas",  "Minimizar el tiempo que se mantienen los locks"),
    ("SELECT FOR UPDATE",     "Declarar intención de escritura al inicio (PG/MySQL)"),
    ("Timeout en lock",       "innodb_lock_wait_timeout (MySQL), lock_timeout (PG)"),
    ("Retry con backoff",     "Si hay deadlock, reintentar con espera exponencial"),
]
print("Estrategias de prevención:")
for nombre, desc in estrategias:
    print(f"  • {nombre:<22} → {desc}")

print()
# Patrón retry (recomendado en producción)
print("Patrón retry para deadlocks:")
print("""
  MAX_REINTENTOS = 3
  for intento in range(MAX_REINTENTOS):
      try:
          conn.execute("BEGIN IMMEDIATE")
          # ... operaciones ...
          conn.commit()
          break
      except sqlite3.OperationalError as e:
          conn.rollback()
          if intento < MAX_REINTENTOS - 1:
              time.sleep(0.1 * (2 ** intento))  # backoff exponencial
          else:
              raise
""")

## Transferencia Bancaria Atómica
> Ejemplo completo: crear tabla de cuentas, fondear dos cuentas, ejecutar transferencia con BEGIN/COMMIT, probar rollback por saldo insuficiente y verificar integridad de balances.


In [ ]:
import sqlite3

def crear_banco():
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE cuentas (
        id INTEGER PRIMARY KEY,
        titular TEXT NOT NULL,
        saldo REAL NOT NULL,
        CHECK(saldo >= 0)
    )''')
    cursor.executemany('INSERT INTO cuentas VALUES (?,?,?)', [
        (1, 'Alice', 2000.00),
        (2, 'Bob',    500.00),
        (3, 'Carol',  150.00),
    ])
    conn.commit()
    return conn

def mostrar_cuentas(conn, titulo="Estado de cuentas"):
    cursor = conn.cursor()
    print(f"\n── {titulo} ──")
    total = 0
    for row in cursor.execute("SELECT id, titular, saldo FROM cuentas ORDER BY id"):
        print(f"  #${row[0]} ${row[1]:<8}: $${row[2]:>8,.2f}")
        total += row[2]
    print(f"  {'TOTAL':<12}: ${total:>8,.2f}")

def transferir(conn, id_origen, id_destino, monto, descripcion=""):
    cursor = conn.cursor()
    print(f"\n{'='*50}")
    print(f"Transferencia: #{id_origen} → #{id_destino}  ${monto:.2f}  {descripcion}")
    try:
        conn.execute("BEGIN")

        # Obtener y verificar saldo origen
        row = cursor.execute(
            "SELECT titular, saldo FROM cuentas WHERE id = ?", (id_origen,)
        ).fetchone()
        if not row:
            raise ValueError(f"Cuenta #{id_origen} no existe")
        titular_origen, saldo = row

        if saldo < monto:
            raise ValueError(
                f"Saldo insuficiente en {titular_origen}: ${saldo:.2f} disponible, ${monto:.2f} solicitado"
            )

        # Obtener datos destino
        row_dest = cursor.execute(
            "SELECT titular FROM cuentas WHERE id = ?", (id_destino,)
        ).fetchone()
        if not row_dest:
            raise ValueError(f"Cuenta destino #{id_destino} no existe")

        # Débito y crédito atómicos
        cursor.execute("UPDATE cuentas SET saldo = saldo - ? WHERE id = ?", (monto, id_origen))
        cursor.execute("UPDATE cuentas SET saldo = saldo + ? WHERE id = ?", (monto, id_destino))

        conn.commit()
        print(f"✅ Éxito: ${monto:.2f} de {titular_origen} → {row_dest[0]}")
        return True

    except Exception as e:
        conn.rollback()
        print(f"❌ Revertida: {e}")
        return False

# ── Ejecución ──
conn = crear_banco()
mostrar_cuentas(conn, "Estado inicial")

# Transferencias válidas
transferir(conn, 1, 2, 300.00, "(Alice paga a Bob)")
transferir(conn, 2, 3, 200.00, "(Bob paga a Carol)")

mostrar_cuentas(conn, "Tras transferencias válidas")

# Transferencias inválidas
transferir(conn, 3, 1, 500.00, "(Carol intenta enviar más de lo que tiene)")
transferir(conn, 9, 1, 100.00, "(cuenta inexistente)")

mostrar_cuentas(conn, "Estado final (sin cambios por las inválidas)")

# Verificar consistencia: suma total invariante
total = conn.execute("SELECT SUM(saldo) FROM cuentas").fetchone()[0]
print(f"\n🔍 Suma total: ${total:.2f} — {'✅ Consistente' if total == 2650.0 else '❌ ERROR'}")

## Batch Insert con SAVEPOINT
> Insertar 100 productos en grupos de 10 usando SAVEPOINT antes de cada grupo. Simular 2 grupos fallidos. Solo los grupos fallidos se revierten; el resto se confirma.

In [ ]:
import sqlite3
import random

random.seed(42)

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL UNIQUE,
    precio REAL NOT NULL CHECK(precio > 0),
    categoria TEXT NOT NULL,
    stock INTEGER NOT NULL DEFAULT 0
)''')
conn.commit()

# Generar 100 productos en 10 grupos de 10
categorias = ['Electronics', 'Furniture', 'Office', 'Clothing', 'Sports']
grupos = []
for g in range(10):
    grupo = []
    for p in range(10):
        idx = g * 10 + p
        nombre = f"Producto-{idx:03d}"
        precio = round(random.uniform(9.99, 999.99), 2)
        categoria = categorias[idx % len(categorias)]
        stock = random.randint(0, 100)
        grupo.append((nombre, precio, categoria, stock))
    grupos.append(grupo)

# Simular fallos: grupo 3 → precio negativo, grupo 7 → nombre duplicado
grupos[3][5] = ("Producto-Falla-A", -50.00, "Office", 10)   # precio inválido
grupos[7][2] = ("Producto-000",      99.99, "Sports", 5)     # nombre ya existe (Producto-000 ya fue en grupo 0)

print(f"Iniciando batch insert: {len(grupos)} grupos × 10 productos = {len(grupos)*10} total")
print()

conn.execute("BEGIN")

resumen = []
for i, grupo in enumerate(grupos):
    sp = f"savepoint_grupo_{i}"
    conn.execute(f"SAVEPOINT {sp}")
    try:
        cursor.executemany(
            "INSERT INTO productos (nombre, precio, categoria, stock) VALUES (?,?,?,?)",
            grupo
        )
        conn.execute(f"RELEASE {sp}")
        resumen.append((i, True, len(grupo), None))
    except sqlite3.IntegrityError as e:
        conn.execute(f"ROLLBACK TO {sp}")
        conn.execute(f"RELEASE {sp}")
        resumen.append((i, False, len(grupo), str(e)[:50]))

conn.commit()

# Mostrar resumen
print(f"{'Grupo':<7} {'Estado':<8} {'Prods':<6} {'Detalle'}")
print("-" * 60)
for i, ok, n, err in resumen:
    estado = "✅ OK" if ok else "❌ FAIL"
    detalle = "" if ok else f"→ {err}"
    print(f"  G{i:<4} {estado:<8} {n:<6} {detalle}")

# Estadísticas finales
total_bd = cursor.execute("SELECT COUNT(*) FROM productos").fetchone()[0]
total_esperado = sum(n for _, ok, n, _ in resumen if ok)
grupos_ok = sum(1 for _, ok, _, _ in resumen if ok)
grupos_fail = sum(1 for _, ok, _, _ in resumen if not ok)

print(f"\n{'='*60}")
print(f"Grupos exitosos  : {grupos_ok}/{len(grupos)}")
print(f"Grupos revertidos: {grupos_fail}/{len(grupos)}")
print(f"Productos en BD  : {total_bd} (esperado: {total_esperado})")
print(f"SAVEPOINT demo   : solo fallaron los grupos con datos inválidos")

# Top 5 por precio
print("\nTop 5 productos más caros:")
for row in cursor.execute("SELECT nombre, precio, categoria FROM productos ORDER BY precio DESC LIMIT 5"):
    print(f"  ${row[0]:<16} $${row[1]:>7.2f}  ${row[2]}")

## Tips y Mejores Prácticas

> Usa siempre transacciones explícitas para operaciones multi-sentencia. El autocommit puede parecer cómodo, pero si el proceso falla a mitad de una operación compuesta, tus datos quedan en un estado inconsistente.


> Mantén las transacciones lo más cortas posible. Las transacciones largas bloquean recursos y aumentan el riesgo de deadlocks y conflictos de concurrencia. Haz el trabajo en memoria primero y luego ejecuta la transacción rápido.


> Siempre prueba los paths de rollback, no solo el happy path. Una transacción que "siempre funciona" en tests pero no maneja errores correctamente puede corromper datos en producción.


> Usa el context manager de Python para conexiones SQLite: "with sqlite3.connect(...) as conn" hace automáticamente COMMIT al salir del bloque sin excepción, o ROLLBACK si hay excepción. Es más limpio que try/except/finally manual.

## Errores Comunes

### Olvidar hacer COMMIT (datos perdidos en caso de crash)

¿Por qué ocurre?
- Sin COMMIT explícito, los cambios de la transacción no se persisten en disco. Si el proceso termina o falla antes del commit, todos los cambios se pierden aunque el código no haya lanzado ningún error.

Solución
- Siempre asegúrate de llamar conn.commit() al final de una transacción exitosa. Usa el context manager "with conn:" en sqlite3 que hace commit automático al salir del bloque.

### Capturar excepciones sin hacer ROLLBACK

¿Por qué ocurre?
- Si capturas una excepción y no haces rollback, la transacción queda abierta con estado parcial. Las siguientes operaciones se ejecutan dentro de esa transacción rota, generando resultados impredecibles.

Solución
- En el except, siempre llama conn.rollback() antes de hacer cualquier otra cosa: try: ... except Exception: conn.rollback(); raise — o al menos maneja el error limpiamente.

### Intentar anidar transacciones con BEGIN dentro de BEGIN

¿Por qué ocurre?
- SQL estándar (y SQLite) no soporta transacciones anidadas reales. Un segundo BEGIN dentro de una transacción activa puede causar un error o comportamiento inesperado dependiendo del motor.

Solución
- Usa SAVEPOINT para simular transacciones anidadas: SAVEPOINT nombre, luego ROLLBACK TO nombre o RELEASE nombre. SAVEPOINT sí está soportado dentro de una transacción activa.